# RealThor v0.1 — Solution

This is the complete solution for the Module 1 exercise. The system prompt is loaded from `prompts/system_prompt.md` rather than written inline.

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = os.getenv("LLM_PROVIDER", "openai")
if provider == "vocareum":
    client = OpenAI(base_url="https://openai.vocareum.com/v1", api_key=os.getenv("VOCAREUM_API_KEY"))
else:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
demand = pd.read_csv("data/demand_signals.csv")
supply = pd.read_csv("data/supply_snapshot.csv")

In [ ]:
# Load the prompt from the versioned file
with open("prompts/system_prompt.md") as f:
    SYSTEM_PROMPT = f.read()

print("Prompt loaded from prompts/system_prompt.md")

In [ ]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]

    if d.empty or s.empty:
        return "No data available for the requested segment."

    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [ ]:
def ask_realthor(region: str, property_type: str, bedrooms: int) -> str:
    context = build_context(region, property_type, bedrooms)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ],
    )
    return response.choices[0].message.content

In [ ]:
recommendation = ask_realthor(region="west", property_type="apartment", bedrooms=2)

In [ ]:
print(recommendation)